# Target C: alarm within the next six hours
Predict **yes/no** at each 10-minute interval end using only current and past data.
An alarm at 13:03 labels 13:00 as yes. The future window is `(t, t + 6 hours]`.
Inputs combine current sensors, trailing 1-hour/24-hour/7-day summaries and past alarms.
This experiment is separate from targets A/B. Read `docs/TARGET_C.md` for coverage and evaluation rules.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook inside the repository.")
COVERAGE = ROOT / "data" / "alarm_coverage.csv"
RUN_BUILD = False
RUN_MODEL = False

## Build and fit
Prepare cleaned SCADA, raw alarm logs and a verified coverage CSV first.
Set the controls above to run. The model fits on configured train years and evaluates
validation years only. Official test years are not scored.

In [ ]:
if RUN_BUILD:
    subprocess.run(
        [sys.executable, str(ROOT / "scripts/build_target_c.py"), "--coverage", str(COVERAGE)],
        cwd=ROOT,
        check=True,
    )
if RUN_MODEL:
    subprocess.run([sys.executable, str(ROOT / "scripts/run_target_c.py")], cwd=ROOT, check=True)

In [ ]:
results = ROOT / "reports/target_c"
if (results / "validation_metrics.json").exists():
    print(json.dumps(json.loads((results / "validation_metrics.json").read_text()), indent=2))
    print(pd.read_csv(results / "validation_predictions.csv").head(20).to_string(index=False))
else:
    print("No results yet. Prepare the source data, then enable build and model controls.")

## Local performance run: 2026-10-30
Hourly prediction anchors, with history computed from all 10-minute source readings.
Fit on a uniform sample of 100,000 of 729,996 eligible train anchors (2016–2018, 2020).
Evaluated all 352,552 eligible validation anchors (2021–2022); test years were not scored.

| Metric | Logistic model | Recent-alarm baseline |
|---|---:|---:|
| Average precision | 0.852 | 0.810 |
| Precision | 84.0% | 84.2% |
| Recall | 61.6% | 36.4% |
| Balanced accuracy | 63.6% | 58.2% |

At the fixed 0.5 threshold, model accuracy is 62.6%, below always yes (74.6%).
Only 36.8% of no predictions are correct. There is ranking signal, but the binary
decisions need improvement. Average precision is not percent correct.

**Provisional coverage:** monthly exports are assumed complete. Seven fleet-silent
days in May 2021 and overlapping windows were excluded. Six-hour outcomes overlap,
so these are dependent prediction-point metrics, not event-level detection rates.
See `reports/target_c/PERFORMANCE.md` for the report and chart.